# Pandas Overview

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.datasets import load_iris

In [ ]:
REPO_ROOT = Path.cwd().parents[1]
DATA_DIR = REPO_ROOT / "data"

## Series
<hr/>

#### Intuition

1-dimensional labeled array

#### Initialization

- **list**: values, indices
- **dict**: keys -> indices, values -> values

In [ ]:
my_list = np.array([i for i in range(5)])
indices = [chr(c) for c in range(ord('a'), ord('a') + 5)]

series = pd.Series(my_list, index=indices, dtype=np.int32,)
print(series)

other_indices = [chr(c) for c in range(ord('A'), ord('A') + 5)]
my_dict = {name: value for (name, value) in zip(other_indices, my_list * 2)}

series1 = pd.Series(my_dict, dtype=np.int32)
print(series1)

#### Element accessing

- **loc**: provide id (location by label)
- **iloc**: provide position (int location)

In [ ]:
print(series['a']) # 0
print(series.loc['b']) # 1
print(series.iloc[2]) # 2
print("Filter by value")
print(series[series > 2])


## Dataframe

#### Intuition

Tabular data structure with rows & columns like 2-dimensional array

#### Initialization

- **dict[Any, list[Any]]**: keys -> columns, values -> items
- **np.array**: 2 dimensional numpy array 
- files like **csv**, **json** etc
- **sklearn datasets**

In [ ]:
my_dict = {
    "course": ["DiffEq", "ProgTech", "Circuits"],
    "grade": [8, 10, 10],
}

grades = pd.DataFrame(my_dict)
grades

In [ ]:
# important for reproducibility
np.random.seed(42)

# data
array = np.random.random((3,4))
columns = np.array(["a", "b", "c", "d"])
rows = np.array([1, 2, 3])

df = pd.DataFrame(data=array, index=rows, columns=columns)
df

In [ ]:
pd.read_csv(DATA_DIR / "mini_csv.csv", index_col="ID")

In [ ]:
dataset = load_iris()
iris_df = pd.DataFrame(dataset.data, columns=dataset.feature_names)
iris_df

#### Quick data overview

- **head**: gives the first 5 rows (similar: *tail*, *sample*)
- **info**: general info 
- **describe**: returns important statistics
- **columns**: column names
- **index**: row names
- **shape**
- **dtypes**: per column data types  
- **corr**: correlation matrix *important for feature selection*
- **isna**: returns boolean df (similar: *notna*, useful: *df.isna().sum()* )

In [ ]:
iris_df.head()

In [ ]:
iris_df.info()

In [ ]:
iris_df.describe()

In [ ]:
print(f"Column names: {iris_df.columns}")

In [ ]:
print(f"Row names: {iris_df.index}")

In [ ]:
print(f"Shape: {iris_df.shape}")

In [ ]:
print(f"Data types: {iris_df.dtypes}")

In [ ]:
iris_df.corr()

#### Element accessing

- **operator[]**
- **loc**
- **iloc**

Note that we can use:
- loc[*row_indexer*, *column_indexer*]
- lists as indexers
- slicing

In [ ]:
grades["course"]

In [ ]:
grades[["course", "grade"]]

In [ ]:
grades.loc[:, "course"]

In [ ]:
# column_indexer is ommited => : 
# this doesn't work for row_indexer
grades.loc[0]

#### Mutability

- add / remove columns / rows
- mutate records

*Always assign*

In [ ]:
# new col
grades["difficulty"] = np.array([5, 4, 3])
grades

In [ ]:
# new row
new_row = pd.DataFrame([{
    "course": "Semiconductors",
    "grade": 9,
    "difficulty": 3,
}], # list object indicates that there could be more rows to add
index=[3]) # always specify, since default is 0
grades = pd.concat([grades, new_row])

grades

In [ ]:
# record edit
grades.loc[3, "grade"] = 10
grades

In [ ]:
courses = grades.drop(columns=["grade"])
courses

#### Filtering

df[ df[ *features* ] matches conditions]

c style logical operators *( &, | )*

In [ ]:
grades[grades["difficulty"] > 3]

#### Aggregate functions

Summarize a set of values

Examples:<br/>
mean, median, std, min, mode, max, sum, count, value_counts

In [ ]:
grades.mean(numeric_only=True)

In [ ]:
grades["difficulty"].value_counts()

Groupby

In [ ]:
group = grades.groupby("difficulty")
group["grade"].mean()

#### Data Cleaning

- *drop*: drops specified cols / rows
- *dropna*: drops the cols / rows with na (not available) values
- *fillna*: fills na with input
- *drop_duplicates*: drops duplicate rows

In [ ]:
grades.loc[4, "course"] = "Analysis"
grades


Note that ints were converted to floats.<br/>
That is caused by the addition of NaN values because pandas cannot store them as int

*Reminder*: The following lines don't change the original df because we have to assign

important **dropna** args:
- how: 'any' | 'all' => if `how`(na) drop
- thresh: at least `thresh` na vals to drop
- subset: select cols / rows

In [ ]:
grades.dropna(subset="grade")

Common fill values:
- 0 / -1
- None
- mean / median / mode

In [ ]:
grades.fillna({"grade": -1}) # fillna(-1) fills all nan values

Fill functions:
- **ffill**: forward fill
    - Copies the *last* valid value forward to next missing rows.
    - Sequential data where values stay the same until they change (e.g., status, price tiers).
- **bfill**: backward fill
    - Copies the *next* valid value backward to previous missing rows.
    - Situations where you need to look ahead to fill current gaps.
- **interpolation**
    - Estimates values by drawing a line or curve between surrounding valid points.
    - Continuous, numeric data with mathematical trends (e.g., temperature, stock trends).

#### Transformation functions

- **apply**: applies a function along an axis
- **map**: applies a function element-wise
- **clip**: caps outliers by limiting values to a specified range
- **pd.cut**: Segments data into equal-*width* intervals
- **pd.qcut**: Segments data into equal-*size* intervals (quantiles)
- **pd.get_dummies**: one hot encoding
- **pd.factorize**: label encoding

In [ ]:
mean_diff = courses.loc[:, "difficulty"].mean()
courses["difficulty"] = courses["difficulty"].map(lambda d : 'hard' if d > mean_diff else 'easy')
# or: courses["difficulty"] = np.where(courses["difficulty"] > mean_diff, 'hard', 'easy')
courses

In [ ]:
pd.cut(iris_df["sepal length (cm)"], bins=3, labels=["low", "mid", "high"]).head()